# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a demonstration for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access basic metadata
print(f"Dataset title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}, Published: {dataset.metadata.datePublished}")
if hasattr(dataset.metadata, 'keywords'):
    print("Keywords:", dataset.metadata.keywords)


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their `@id`s
record_set_ids = dataset.record_sets
print("Available record sets (by @id):")
for rs in record_set_ids:
    print(f"  - {rs}")

# Preview fields in the first record set
if record_set_ids:
    fields = dataset.fields(record_set=record_set_ids[0])
    print(f"\nFields for record set {record_set_ids[0]}:")
    for f in fields:
        print(f"  - @id: {f['@id']}, name: {f['name']}, dataType: {f.get('dataType', 'NA')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Define the record set(s) we want to extract (fill in with IDs from previous output)
record_sets_to_load = record_set_ids[:1]  # Load the first record set for demonstration
dataframes = {}

for rs_id in record_sets_to_load:
    rows = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(rows)
    dataframes[rs_id] = df
    print(f"Loaded data for record set {rs_id}: shape={df.shape}")
    print("Sample columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Select a numeric field for analysis
# You should use the '@id' of the field; here, let's try to infer a suitable field from the columns
sample_rs_id = record_sets_to_load[0]
numeric_field_candidates = [col for col in dataframes[sample_rs_id].columns if pd.api.types.is_numeric_dtype(dataframes[sample_rs_id][col])]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]  # Pick the first numeric field found
else:
    print("No clear numeric field found. Please adjust field selection based on the dataset.")
    numeric_field = dataframes[sample_rs_id].columns[0]

# Filter records with value greater than a threshold
threshold = dataframes[sample_rs_id][numeric_field].mean() if pd.api.types.is_numeric_dtype(dataframes[sample_rs_id][numeric_field]) else 0
filtered_df = dataframes[sample_rs_id][dataframes[sample_rs_id][numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[numeric_field + "_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

# Optionally group by a categorical field
categorical_candidates = [col for col in filtered_df.columns if filtered_df[col].dtype == 'object']
if categorical_candidates:
    group_field = categorical_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())
else:
    print("No suitable categorical group field available for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in dataframes[sample_rs_id].columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[sample_rs_id][numeric_field].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

if len(numeric_field_candidates) > 1:
    y_field = numeric_field_candidates[1]
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=dataframes[sample_rs_id][numeric_field], y=dataframes[sample_rs_id][y_field])
    plt.title(f'{numeric_field} vs {y_field}')
    plt.xlabel(numeric_field)
    plt.ylabel(y_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset contains protein information from mass spectrometry analysis of extracellular vesicles from human mast cells.
- We loaded and inspected the data using the `mlcroissant` library, filtered by a numeric attribute, and visualized value distributions.
- For further biological analysis, you can repeat these steps for other record sets and fields, referencing their `@id`s.